In [72]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [73]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [74]:
!pip install nltk



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [75]:
!pip install package_name



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
!pip install pandas




[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [77]:
!pip install faiss-cpu




[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
!pip install sentence-transformers -i https://pypi.org/simple


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [79]:
!pip install --upgrade pip
!pip install --no-cache-dir sentence-transformers

  Using cached pip-25.1-py3-none-any.whl.metadata (3.6 kB)
Using cached pip-25.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\Hp\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [80]:
pip install sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [81]:
pip install faiss-cpu -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [82]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sentence_transformers import SentenceTransformer
import torch
from sentence_transformers.util import cos_sim


In [83]:
# Load FAQ CSV
df = pd.read_csv("Dataset.csv", encoding='ISO-8859-1')

In [84]:
print(df.head())

                                            Question  \
0  What is the tuition fee per session for Medica...   
1  What is the tuition fee per semester for Medic...   
2  Can I apply to study Medical Laboratory Scienc...   
3  Is Medical Laboratory Science offered in the s...   
4  What are the requirements to apply for Medical...   

                                              Answer  
0  The tuition fee for Medical Laboratory Science...  
1  The tuition fee for Medical Laboratory Science...  
2  Yes, Medical Laboratory Science is one of the ...  
3   Yes, we offer Medical Laboratory Science at NEU.  
4  To apply for Medical Laboratory Science, you n...  


In [85]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Question  224 non-null    object
 1   Answer    224 non-null    object
dtypes: object(2)
memory usage: 3.6+ KB
None


In [86]:
print(df.describe())

                                                 Question  \
count                                                 224   
unique                                                224   
top     What is the tuition fee per session for Medica...   
freq                                                    1   

                                                   Answer  
count                                                 224  
unique                                                224  
top     The tuition fee for Medical Laboratory Science...  
freq                                                    1  


In [87]:
print(df.isnull().sum())

Question    0
Answer      0
dtype: int64


In [88]:
print(df.nunique())

Question    224
Answer      224
dtype: int64


In [89]:
#Remove duplicate questions (keep the first occurrence)
df = df.drop_duplicates(subset="Question", keep="first")

In [90]:
#Trim whitespace and remove question marks from the 'Question' column
df["Question"] = df["Question"].str.strip().str.replace("?", "", regex=False)

#Strip whitespace from 'Answer' column as well
df["Answer"] = df["Answer"].str.strip()

In [91]:
#Capitalize the first letter of each question and answer
df["Question"] = df["Question"].str.capitalize()
df["Answer"] = df["Answer"].str.capitalize()
print(df.head())

                                            Question  \
0  What is the tuition fee per session for medica...   
1  What is the tuition fee per semester for medic...   
2  Can i apply to study medical laboratory scienc...   
3  Is medical laboratory science offered in the s...   
4  What are the requirements to apply for medical...   

                                              Answer  
0  The tuition fee for medical laboratory science...  
1  The tuition fee for medical laboratory science...  
2  Yes, medical laboratory science is one of the ...  
3   Yes, we offer medical laboratory science at neu.  
4  To apply for medical laboratory science, you n...  


In [92]:
 #Preprocessing Function (for NLP)
stop_words = set(stopwords.words("english"))

In [93]:
def preprocess_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # remove punctuation
    words = word_tokenize(text)  # tokenize
    words = [word for word in words if word not in stop_words]  # remove stopwords
    return " ".join(words)  # rejoin

In [94]:
#  Apply Preprocessing to Question Column to improve keyword search and others
df["Preprocessed_Question"] = df["Question"].apply(preprocess_text)

# Save the Cleaned and Preprocessed Dataset
df.to_csv("Dataset2.csv", index=False)

# Preview
df[["Question", "Preprocessed_Question", "Answer"]].head()

,Question,Preprocessed_Question,Answer
0,What is the tuition fee per session for medica...,tuition fee per session medical laboratory sci...,The tuition fee for medical laboratory science...
1,What is the tuition fee per semester for medic...,tuition fee per semester medical laboratory sc...,The tuition fee for medical laboratory science...
2,Can i apply to study medical laboratory scienc...,apply study medical laboratory science neu,"Yes, medical laboratory science is one of the ..."
3,Is medical laboratory science offered in the s...,medical laboratory science offered school,"Yes, we offer medical laboratory science at neu."
4,What are the requirements to apply for medical...,requirements apply medical laboratory science,"To apply for medical laboratory science, you n..."


In [103]:
# Load a pretrained model for embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast & effective for Q&A

# Compute embeddings for all questions
question_embeddings = model.encode(df["Preprocessed_Question"].tolist(), convert_to_tensor=True)

# Sample query
query = "what is the deadline for admission"
query_embedding = model.encode(query, convert_to_tensor=True)

# Compute cosine similarity
scores = cos_sim(query_embedding, question_embeddings)

# Get top match
top_idx = scores.argmax()
best_match = df.iloc[top_idx.item()]

# Show result
print("User query:", query)
print("Matched question:", best_match["Preprocessed_Question"])
print("Answer:", best_match["Answer"])

User query: what is the deadline for admission
Matched question: admission still open year
Answer: Yes, admission for the 2024/2025 academic session is currently open.


# Saving our Model

In [104]:
class SemanticChatbot:
    def __init__(self, df, index, model):
        self.df = df
        self.index = index
        self.model = model

    def get_response(self, query):
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        question_embeddings = self.model.encode(self.df["Preprocessed_Question"].tolist(), convert_to_tensor=True)
        scores = cos_sim(query_embedding, question_embeddings)
        top_idx = scores.argmax()
        best_match = self.df.iloc[top_idx.item()]
        return best_match["Answer"]

In [105]:
chatbot = SemanticChatbot(df=df, index=index, model=model)

# Save using joblib
import joblib
joblib.dump(chatbot, 'chatbot_model.pkl')

['chatbot_model.pkl']